# Induction Heads in GPT-2 Small — Kaggle worksheet

**Goal:** locate the *induction heads* in GPT-2 and prove *causally* that they drive in-context copying.

**Before you start:** Notebook settings → Accelerator = **GPU T4**, Internet = **ON**.

**How to use this:** each STEP has an explanation + a code stub with `# TODO`. *You* write the code — that's the whole point (this is the interp skill you're building, and the finished notebook is your portfolio artifact). Run each step, read the result, then move on. Don't peek at the ARENA solutions.

**Plan:** 1) phenomenon → 2) localise heads → 3) attention pattern → 4) causal ablation → 5) the circuit.

In [ ]:
# --- setup: install the interpretability libs (torch is already on Kaggle) ---
!pip install -q transformer_lens circuitsvis

In [ ]:
import torch, einops
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer
import circuitsvis as cv

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2-small", device=device)
print("Loaded gpt2-small on", device, "| layers", model.cfg.n_layers,
      "| heads", model.cfg.n_heads, "| d_model", model.cfg.d_model)

## STEP 1 — The phenomenon: in-context copying

Feed the model a random token sequence, then **repeat it**. Induction heads make the model predict the *second* copy far better than the first, so the per-position log-prob of the correct next token **jumps up** right after the repeat boundary. Measuring that jump proves a copying circuit exists (before we hunt for the heads in Step 2).

In [ ]:
# STEP 1 — write this yourself (~8 lines)
# 1) build `tokens`: a BOS token, then the SAME random sequence TWICE
#    (torch.randint over model.cfg.d_vocab for the ids; torch.cat to assemble; shape [1, 1+2*seq_len])
# 2) run the model -> logits -> log-probs; pull the log-prob of the ACTUAL next token at
#    each position (hint: logits.log_softmax(-1), then gather against tokens[:, 1:])
# 3) plot the per-position log-prob; draw a vertical line at x = seq_len
#    -> where does it jump, and by how much?

raise NotImplementedError("Implement STEP 1")

## STEP 2 — Localise the induction heads

Find WHICH of the 144 heads cause that jump. Define an **induction score** per head: on the repeated sequence, how much does each head attend from the current token back to *the token that followed the previous occurrence* — i.e. attention on the diagonal offset by `-(seq_len-1)`? Compute it for every (layer, head) and plot a heatmap. A few heads should light up.

**Key tool:** `logits, cache = model.run_with_cache(tokens)` gives every head's attention pattern in `cache["pattern", layer]` (shape `[batch, head, query, key]`).

In [ ]:
# STEP 2 — write this yourself
# 1) logits, cache = model.run_with_cache(tokens)
# 2) for each layer, grab cache["pattern", layer]           # [batch, head, q, k]
# 3) induction score for a head = mean attention on the diagonal offset by -(seq_len-1)
#    (attention from query position q back to key position q-(seq_len-1); torch.diagonal helps)
# 4) collect into a [n_layers, n_heads] tensor, imshow it, and note the top head(s)

raise NotImplementedError("Implement STEP 2")

## STEP 3 — What is the top head doing?

Visualise the top induction head's attention on the repeated sequence with **circuitsvis**. You should see each token in the second copy attending back to *the token right after its first occurrence* — the copy mechanism made visible.

`cv.attention.attention_patterns(tokens=str_tokens, attention=cache["pattern", layer][0])`

In [ ]:
# STEP 3 — write this yourself
# 1) str_tokens = model.to_str_tokens(tokens)
# 2) render the top induction head's pattern with cv.attention.attention_patterns(...)
#    (slice cache["pattern", top_layer] to your head)

raise NotImplementedError("Implement STEP 3")

## STEP 4 — Prove it causally (ablation)

Correlation isn't causation. **Ablate** the induction head(s) — zero their output via a hook — and re-measure loss on the repeated sequence. If loss on the second copy spikes back up, the heads *cause* the copying.

Use `model.run_with_hooks(tokens, fwd_hooks=[(name, hook_fn)])`, where `hook_fn` zeros the head's `z` at the relevant `blocks.{layer}.attn.hook_z`.

In [ ]:
# STEP 4 — write this yourself
# 1) define a hook that sets a specific head's z (or result) to 0
# 2) run_with_hooks on the induction head(s); compute loss on the 2nd-copy positions
# 3) compare loss WITH vs WITHOUT ablation -> it should jump up when ablated

raise NotImplementedError("Implement STEP 4")

## STEP 5 — The circuit: previous-token head → induction head

The induction head only works because an earlier **previous-token head** (usually in layer 0) writes "what token came before me" into the residual stream. Find that prev-token head (attention on the `-1` diagonal), and optionally ablate it to show the induction head breaks. That completes the two-step circuit.

In [ ]:
# STEP 5 — write this yourself
# 1) compute a "previous-token score" per head (attention on the diagonal offset -1)
# 2) identify the layer-0 prev-token head
# 3) (optional) ablate it and show the induction score of the induction head collapses

raise NotImplementedError("Implement STEP 5")

## Done?

Once all five run, you have: the phenomenon, the responsible heads, the mechanism, the causal proof, and the circuit. Export your figures, fill in `WRITEUP.md`, then **download this notebook and commit it to the repo** — that's your publishable interpretability write-up.